# AIDE Inference Experiment

## Command Mapping

This notebook runs clean inference equivalent to the original AIDE eval command:

```bash
python main_finetune.py \
    --data_path dataset/progan/train \
    --eval_data_path dataset/progan/eval \
    --resume results/progan_train/GenImage_train.pth \
    --eval True \
    --output_dir results/progan_train
```

**What changed**: Training loop, DDP, Mixup, EMA, optimizer stripped out. Inference-only pipeline with per-image score output.

## Datasets

- **NTIRE**: 2,500 test images with binary labels (0=authentic, 1=fake)
- **Z-Image-Turbo**: 100 AI-generated images (all label=1)

## Reproducibility

- Seed: 42
- Checkpoint: `GenImage_train.pth`
- batch_size: 4 (WSL VRAM-safe)
- Deterministic: `torch.manual_seed`, `cudnn.deterministic`

In [ ]:
import json
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.backends.cudnn as cudnn
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)

from diffguard.aide_inference import AIDEDataset, AIDEInferenceRunner

# --- Reproducibility ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
cudnn.deterministic = True
cudnn.benchmark = False

# --- Paths ---
REPO_ROOT = Path("..").resolve() if not Path("data").exists() else Path(".")
CHECKPOINT = REPO_ROOT / "data" / "artifacts" / "checkpoints" / "GenImage_train.pth"
NTIRE_IMAGES = REPO_ROOT / "data" / "bronze" / "ntire" / "test_images"
NTIRE_LABELS = REPO_ROOT / "data" / "bronze" / "ntire" / "test_labels.csv"
ZIT_IMAGES = REPO_ROOT / "data" / "bronze" / "z_image_turbo"
ZIT_METADATA = REPO_ROOT / "data" / "bronze" / "z_image_turbo" / "metadata.csv"
OUTPUT_DIR = REPO_ROOT / "data" / "silver" / "aide"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "plots").mkdir(exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 4

print(f"PyTorch {torch.__version__}")
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"Checkpoint: {CHECKPOINT} (exists={CHECKPOINT.exists()})")

In [ ]:
# Model init — only needed for re-running inference.
# Set RE_INFERENCE=False to load saved results from data/silver/aide/.
RE_INFERENCE = False

if RE_INFERENCE:
    runner = AIDEInferenceRunner(
        checkpoint_path=str(CHECKPOINT),
        device=DEVICE,
        batch_size=BATCH_SIZE,
    )
    print(f"Model loaded. bf16={runner.use_bf16}")
else:
    print(
        "Loading saved results from data/silver/aide/ (set RE_INFERENCE=True to re-run)"
    )

In [ ]:
# --- NTIRE Results ---
results_ntire = pd.read_csv(OUTPUT_DIR / "ntire_scores.csv")
with open(OUTPUT_DIR / "ntire_metrics.json") as f:
    metrics_ntire = json.load(f)

acc = metrics_ntire["accuracy"]
ap = metrics_ntire["average_precision"]
auc = metrics_ntire["auc_roc"]

print(f"NTIRE Results ({len(results_ntire)} images):")
print(f"  Accuracy: {acc:.4f}")
print(f"  Average Precision: {ap:.4f}")
print(f"  AUC-ROC: {auc:.4f}")
print(f"\nScore distribution by label:")
for lbl in [0, 1]:
    subset = results_ntire[results_ntire["label"] == lbl]
    tag = "authentic" if lbl == 0 else "fake"
    print(
        f"  {tag} (n={len(subset)}): mean={subset['score'].mean():.4f}, median={subset['score'].median():.4f}"
    )

In [ ]:
# --- Z-Image-Turbo Results ---
results_zit = pd.read_csv(OUTPUT_DIR / "z_image_turbo_scores.csv")

print(f"Z-Image-Turbo Results ({len(results_zit)} images, all label=1):")
print(f"  Mean score: {results_zit['score'].mean():.4f}")
print(f"  Median score: {results_zit['score'].median():.4f}")
print(f"  Score > 0.5: {(results_zit['score'] > 0.5).sum()} / {len(results_zit)}")
print(f"  Score > 0.8: {(results_zit['score'] > 0.8).sum()} / {len(results_zit)}")

In [ ]:
from IPython.display import SVG, display
from sklearn.metrics import precision_recall_curve, roc_curve

y_true = results_ntire["label"].values
y_score = results_ntire["score"].values

# --- Plot 1: Score histogram ---
fig1, ax = plt.subplots(figsize=(6, 4))
for lbl, color, tag in [
    (0, "steelblue", "NTIRE authentic"),
    (1, "coral", "NTIRE fake"),
]:
    subset = results_ntire[results_ntire["label"] == lbl]
    ax.hist(subset["score"], bins=40, alpha=0.5, label=tag, color=color, density=True)
ax.hist(
    results_zit["score"],
    bins=25,
    alpha=0.6,
    label="Z-Image-Turbo (all fake)",
    color="darkred",
    density=True,
)
ax.axvline(0.5, color="black", linestyle="--", linewidth=0.8, label="Threshold 0.5")
ax.set_xlabel("Fake Probability Score")
ax.set_ylabel("Density")
ax.set_title("Score Distribution")
ax.legend(fontsize=7)
plt.tight_layout()
fig1.savefig(OUTPUT_DIR / "plots" / "score_histogram.svg", format="svg")
plt.close(fig1)

# --- Plot 2: ROC curve ---
fig2, ax = plt.subplots(figsize=(5, 4))
fpr, tpr, _ = roc_curve(y_true, y_score)
ax.plot(fpr, tpr, color="steelblue", linewidth=2, label=f"AUC = {auc:.4f}")
ax.plot([0, 1], [0, 1], "k--", linewidth=0.8)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title(f"ROC Curve — NTIRE (Acc={acc:.4f})")
ax.legend()
plt.tight_layout()
fig2.savefig(OUTPUT_DIR / "plots" / "ntire_roc_curve.svg", format="svg")
plt.close(fig2)

# --- Plot 3: PR curve ---
fig3, ax = plt.subplots(figsize=(5, 4))
prec, rec, _ = precision_recall_curve(y_true, y_score)
ax.plot(rec, prec, color="steelblue", linewidth=2)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title(f"Precision-Recall — NTIRE (AP={ap:.4f})")
plt.tight_layout()
fig3.savefig(OUTPUT_DIR / "plots" / "ntire_pr_curve.svg", format="svg")
plt.close(fig3)

print(f"Saved SVGs to {OUTPUT_DIR / 'plots'}")
display(SVG(filename=OUTPUT_DIR / "plots" / "score_histogram.svg"))

In [ ]:
display(SVG(filename=OUTPUT_DIR / "plots" / "ntire_roc_curve.svg"))
display(SVG(filename=OUTPUT_DIR / "plots" / "ntire_pr_curve.svg"))

## Summary

| Dataset | Images | Accuracy | AP | AUC-ROC | Mean Score (fake) |
|---------|--------|----------|-----|---------|-------------------|
| NTIRE | 2,500 | 0.5488 | 0.6230 | 0.6127 | — |
| Z-Image-Turbo | 100 | — | — | — | 0.7710 |

**Key findings:**
- GenImage_train checkpoint shows modest generalization to NTIRE (AP=0.62), suggesting the checkpoint was trained on a different distribution of generators.
- AIDE detects 86/100 Z-Image-Turbo images as fake (score > 0.5), with mean score 0.77 — reasonable performance on modern diffusion outputs.
- The NTIRE dataset likely contains generators unseen during GenImage training, explaining the near-chance accuracy.

**Artifacts:**
- `data/silver/aide/ntire_scores.csv` — per-image scores for NTIRE
- `data/silver/aide/z_image_turbo_scores.csv` — per-image scores for Z-Image-Turbo
- `data/silver/aide/ntire_metrics.json` — NTIRE classification metrics
- `data/silver/aide/plots/` — histograms, ROC, PR curves